# M.Tech AIML – Machine Learning Assignment 2
## UCI Dry Bean Classification
Run all cells from top to bottom. This notebook downloads the dataset, trains five classifiers, calculates the required metrics, saves models and exports test_data.csv for Streamlit.

In [ ]:
# If you see NumPy/SciPy import errors, run this cell once, restart the kernel, then run the notebook again.
%pip install --upgrade "numpy>=2.0,<2.8" "scipy>=1.14" pandas matplotlib scikit-learn ucimlrepo joblib streamlit

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, classification_report, ConfusionMatrixDisplay)
import joblib

RANDOM_STATE = 42
os.makedirs("model", exist_ok=True)
print("Imports successful. NumPy:", np.__version__)

In [ ]:
# Download dataset from the UCI Machine Learning Repository
dry_bean = fetch_ucirepo(id=602)
X = dry_bean.data.features.copy().apply(pd.to_numeric, errors="coerce")
y_raw = dry_bean.data.targets
target_col = "Class" if "Class" in y_raw.columns else y_raw.columns[0]
y = y_raw[target_col].astype(str).rename("Class")
df = pd.concat([X, y], axis=1)

print("Dataset shape:", df.shape)
print("Feature count:", X.shape[1])
print("Instance count:", len(df))
print("\\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))
print("\\nClass distribution:")
display(y.value_counts().to_frame("Count"))
display(df.head())

assert X.shape[1] >= 12 and len(df) >= 500, "Dataset does not meet assignment minimums."
df.to_csv("dry_bean_dataset.csv", index=False)
print("Saved dry_bean_dataset.csv")

In [ ]:
# Stratified 80/20 split. All models use this exact same split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## Train and evaluate the five required classifiers
Scaling is applied to Logistic Regression, KNN and SVM. Preprocessing is fitted inside each pipeline using training data only.

In [ ]:
models = {
    "Logistic Regression": Pipeline([("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
    "Decision Tree": Pipeline([("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    "KNN": Pipeline([("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()), ("classifier", KNeighborsClassifier(n_neighbors=5))]),
    "Naive Bayes": Pipeline([("imputer", SimpleImputer(strategy="median")),
        ("classifier", GaussianNB())]),
    "Support Vector Machine": Pipeline([("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()), ("classifier", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))])
}
print("Five models prepared.")

In [ ]:
results = []
trained_models = {}
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "support_vector_machine.joblib"
}

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)
    results.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, prob, multi_class="ovr", average="weighted"),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, pred, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_test, pred)
    })
    trained_models[name] = model
    joblib.dump(model, os.path.join("model", model_files[name]))

results_df = pd.DataFrame(results)
display(results_df.round(4))
results_df.to_csv("model_comparison.csv", index=False)
winner = results_df.loc[results_df["F1 Score"].idxmax()]
print("Winner based on weighted F1 Score:", winner["ML Model Name"])

In [ ]:
# Actual classification report and confusion matrix for the best weighted-F1 model
winner_name = winner["ML Model Name"]
winner_model = trained_models[winner_name]
winner_pred = winner_model.predict(X_test)
print(classification_report(y_test, winner_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(y_test, winner_pred, xticks_rotation=45, cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix - " + winner_name)
plt.tight_layout()
plt.show()

In [ ]:
# Export labelled held-out test data for the Streamlit app
test_data = X_test.copy()
test_data["Class"] = y_test.values
test_data.to_csv("test_data.csv", index=False)

print("Created test_data.csv:", test_data.shape)
print("Also created dry_bean_dataset.csv, model_comparison.csv and five files in model/.")

## Final notes
Use the actual metric values from model_comparison.csv in your assignment report. Write observations based on the measured results. Do not invent scores. The Streamlit app loads these saved pipelines and evaluates uploaded test data without retraining.